任务详情：如果CHOIR分群结果理想，则可以完成metaneighbor和树状图层级聚类分析\
参数列表：没有参数需要调整\
输出结果：\
1.MetaNeighbor相似性矩阵: 保存至qingming_metaNeighbor.csv\
2.MetaNeighbor聚类结果: 保存至qingming_anno.csv\
3.层次聚类结果: 保存至qingming_Dendrogram_Clusters.csv\
4.完整分析结果: 保存至NipLSD_DenC40.rds

In [ ]:
# 数据路径读取
setwd("/data/")

# 加载所需库
library(MetaNeighbor)
library(SummarizedExperiment)
library(Seurat)
library(SingleCellExperiment)
library(grid)
library(ComplexHeatmap)
library(circlize)
library(ggplot2)
library(igraph)
library(plyr)
library(RColorBrewer)

# 读取Seurat对象
seurat_obj <- readRDS("NipLSD_alpha0.5_harmony.rds")
output_name <- "0410_alpha0.5"  # 输出文件的前缀名
batch_key <- "orig.source"
cluster_key <- "CHOIR_clusters_0.05"
threshold_value <- 0.85  # 相似性阈值
sctransform <- FALSE
desired_num_clusters <- 40  # 层次聚类的目标簇数

# 确定使用的assay
if (sctransform) {
  assay <- "SCT"
} else {
  assay <- "RNA"
}
cat(paste0("使用 assay: ", assay, "\n"))

# 转换数据格式
merged_data2 <- seurat_obj
#v5专用：merged_data2[[assay]] <- JoinLayers(merged_data2[[assay]])
sdata <- as.SingleCellExperiment(merged_data2, assay = assay, slot = "counts")

# 计算可变基因
var_genes <- variableGenes(dat = sdata, exp_labels = sdata@colData[[batch_key]])
cat(paste0("找到 ", length(var_genes), " 个可变基因\n"))

# 计算细胞类型相似性
cat("正在计算细胞类型相似性...\n")
celltype_NV <- MetaNeighborUS(
  var_genes = var_genes,
  dat = sdata,
  study_id = sdata@colData[[batch_key]],
  cell_type = sdata@colData[[cluster_key]],
  fast_version = TRUE
)
cat("相似性矩阵计算完成\n")

# 保存相似性矩阵
output_csv <- paste0(output_name, "_metaNeighbor.csv")
write.csv(celltype_NV, file = output_csv, quote = FALSE, row.names = TRUE)
cat(paste0("相似性矩阵已保存到: ", output_csv, "\n"))

# 自动分配一致性注释
threshold <- threshold_value

# 构造邻接矩阵（对角线强制为0，避免自己连自己）
adj <- (celltype_NV >= threshold)
adj[is.na(adj)] <- FALSE
diag(adj) <- 0

# 基于邻接矩阵构造无向图
g <- graph_from_adjacency_matrix(adj, mode = "undirected", weighted = TRUE)

# 提取连通分量
comps <- components(g)

# 生成唯一cluster标签
cluster_df <- data.frame(
  group = names(comps$membership),
  cluster = paste0("cluster_", comps$membership)
)

cat("聚类分配结果:\n")
print(cluster_df)

# 将聚类结果映射回原始数据
group2cluster <- setNames(cluster_df$cluster, cluster_df$group)

# 修复：统一使用|作为分隔符
combined_key <- paste0(batch_key, "|", cluster_key)
seurat_obj@meta.data[[combined_key]] <- paste0(
  seurat_obj@meta.data[[batch_key]],
  "|",
  seurat_obj@meta.data[[cluster_key]]
)
seurat_obj@meta.data$metaneighbor <- group2cluster[seurat_obj@meta.data[[combined_key]]]

# 统计注释分布
cat("\nMetaNeighbor聚类分布:\n")
print(table(seurat_obj@meta.data$metaneighbor))

# 保存注释结果
output_anno_csv <- paste0(output_name, "_anno.csv")
write.csv(
  data.frame(
    barcode = rownames(seurat_obj@meta.data),
    anno = seurat_obj@meta.data$metaneighbor
  ),
  file = output_anno_csv,
  row.names = FALSE
)
cat(paste0("注释结果已保存到: ", output_anno_csv, "\n"))

# 使用MetaNeighbor相似性矩阵进行层次聚类
cat("正在从MetaNeighbor相似性矩阵进行层次聚类...\n")

# 数据清理
celltype_NV_plot <- as.matrix(apply(celltype_NV, 2, as.numeric))
celltype_NV_plot[is.na(celltype_NV_plot)] <- 0
celltype_NV_for_clustering <- (celltype_NV_plot + t(celltype_NV_plot)) / 2

# 计算距离矩阵并聚类
distance_matrix <- as.dist(1 - celltype_NV_for_clustering)
hc_rows <- hclust(distance_matrix, method = "complete")
clusters_from_dendrogram <- cutree(hc_rows, k = desired_num_clusters)

# 创建并保存映射结果（CSV格式）
dendro_cluster_df <- data.frame(
  group = names(clusters_from_dendrogram),
  dendrogram_cluster = paste0("DenC_", clusters_from_dendrogram)
)

# 保存为CSV
csv_output <- paste0(output_name, "_Dendrogram_Clusters.csv")
write.csv(dendro_cluster_df, file = csv_output, row.names = FALSE)
cat(paste0("映射结果已保存到 CSV: ", csv_output, "\n"))

# 修复映射逻辑 - 统一分隔符
# 提取Seurat对象中的原始group值
# 注意：直接使用Seurat对象中已有的"orig.source"和"CHOIR_clusters_0.05"列
# 构建与CSV中完全一致的group格式（如"NipLSD1|1"）
seurat_obj@meta.data$combined_key <- paste(
  seurat_obj@meta.data[[batch_key]], 
  seurat_obj@meta.data[[cluster_key]], 
  sep = "|"  # 确保分隔符与CSV中的group字段一致
)

# 创建映射字典（直接使用CSV中的group值）
group2dendro <- setNames(
  dendro_cluster_df$dendrogram_cluster, 
  dendro_cluster_df$group  # 使用原始group值（如"NipLSD1|1"）
)

# 执行映射（关键：使用combined_key匹配）
seurat_obj@meta.data$dendrogram_40 <- group2dendro[
  seurat_obj@meta.data$combined_key
]

# 验证映射结果
unmapped_cells <- sum(is.na(seurat_obj@meta.data$dendrogram_40))
if(unmapped_cells == 0) {
  cat("成功！所有细胞均已映射，未发现 NA 值。\n")
} else {
  # 诊断未映射的细胞
  problem_ids <- names(which(is.na(group2dendro[
  as.character(unique(seurat_obj@meta.data$combined_key))
])))
  cat("Warning: 仍有映射失败。\n")
  cat("请检查以下 ID 是否存在于 dendro_cluster_df 中:\n")
  print(problem_ids)
}

# 保存更新后的RDS文件
saveRDS(seurat_obj, "NipLSD_DenC40.rds")

cat("分析完成！\n")
cat("- MetaNeighbor相似性矩阵已保存至: ", output_csv, "\n")
cat("- MetaNeighbor聚类结果已保存至: ", output_anno_csv, "\n")
cat("- 层次聚类结果已保存至: ", csv_output, "\n")
cat("- 包含所有结果的Seurat对象已保存至: NipLSD_DenC40.rds\n")


In [ ]:
table(seurat_obj@meta.data$dendrogram_40)

In [ ]:
seurat_obj<-subset(seurat_obj,subset=orig.source=="NipLSD5")

In [ ]:
table(seurat_obj@meta.data$dendrogram_40)